# Sphota GPU run 3: one planned sweep on a free Kaggle GPU (D-050, D-051)

One question decides the positioning: **with a stronger multilingual anchor (bge-m3), do Sphota's fine codes pass the D-017/D-018 decision gate?**
The expensive part (embedding about 0.8M sentences with bge-m3 and 9 other models) is done **once**; on top of it the run evaluates
3 meaning-space variants × 7 adapter variants × 256/512/1024 cells × 2 independent replicates, plus two space widths and a second model panel.
Only one pre-registered combination is the gate test (see `DECISIONS.md` D-051); everything else is reported as exploratory.

**Before you run** (right-hand panel → *Session options*): **Accelerator** GPU T4 ×2 (or P100); **Internet** on. Then *Save Version → Save & Run All*.

**Time:** about 4–5 hours (phase 1 about 3–3.5 h, optional phase 2 about 1 h). A 5–10 minute smoke test runs first and stops the notebook if anything is broken, so a failure costs minutes, not hours.
**Cost:** ₹0 (Kaggle's free weekly GPU quota). **Download:** `sphota-run3.zip` from the Output tab.

In [ ]:
# 1. Get the code (public repository: no token needed, and none should be added)
import os, subprocess, time
os.environ["SPHOTA_DEADLINE"] = str(time.time() + 10.5 * 3600)  # Kaggle stops at 12 h: leave time to save
os.environ["SPHOTA_HF_CACHE"] = "/tmp/sphota-hf"    # models and embedding caches stay out of /kaggle/working
os.environ["SPHOTA_DATA_DIR"] = "/tmp/sphota-data"  # (its 20 GB output limit and the save step)
os.environ["PYTHONUNBUFFERED"] = "1"
if not os.path.exists("/kaggle/working/sphota"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Thiruvikramanpravartak/sphota.git", "/kaggle/working/sphota"], check=True)
os.chdir("/kaggle/working/sphota")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Install (Kaggle already has CUDA torch; this adds the rest)
!pip -q install -e ".[build,encode]"
import torch
os.environ["SPHOTA_THREADS"] = str(os.cpu_count())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on a GPU accelerator")
!df -h /tmp /kaggle/working | tail -2

In [ ]:
# 3. Smoke test (5-10 min): the same sweep code at tiny scale, loading EVERY model the run uses
#    (bge-m3, the 5 gate-panel models, the 4 second-panel models). Stops the notebook if anything fails.
import glob, json
!python -m sphota.build.data --scale tiny 2>&1 | tail -1
!python -m sphota.build.gate v3sweep --scale tiny --grid smoke --anchor BAAI/bge-m3 --sizes 16 32 --n-adapter 3000 2>&1 | tee /kaggle/working/smoke.log | grep -E "^\[|PRIMARY|Error|error" | tail -8
f = "bench/results/v3-sweep-tiny-bge-m3-smoke.json"
ok = os.path.exists(f) and len(json.load(open(f))["rows"]) == 10 and all("cross_model_extra" in r for r in json.load(open(f))["rows"][:4])
os.path.exists(f) and os.remove(f)
if not ok:
    raise RuntimeError("Smoke test FAILED - see /kaggle/working/smoke.log; nothing expensive was run. Send the log.")
print("smoke test OK")

In [ ]:
# 4. PHASE 1 - GPU run 3 (D-050 + D-051), anchor bge-m3, about 3-3.5 h. Partial results are saved after every block.
!python -m sphota.build.data --scale medium 2>&1 | tail -1
!python -m sphota.build.gate v3sweep --scale medium --anchor BAAI/bge-m3 --sizes 256 512 1024 --n-adapter 120000 2>&1 | tee /kaggle/working/run3.log | grep -vE "it/s|sent/s|Batches|Warning|warn"

In [ ]:
# 5. PHASE 2 (exploratory, about 1 h): a second anchor, multilingual-e5-large (same family as the frozen
#    anchor, 24x larger). Reuses every panel embedding from phase 1. Skipped if under 2.5 h remain.
left_h = (float(os.environ["SPHOTA_DEADLINE"]) - time.time()) / 3600
if left_h > 2.5:
    !python -m sphota.build.gate v3sweep --scale medium --grid anchor2 --anchor intfloat/multilingual-e5-large --sizes 256 512 --n-adapter 120000 2>&1 | tee /kaggle/working/run3-anchor2.log | grep -vE "it/s|sent/s|Batches|Warning|warn"
else:
    print(f"phase 2 skipped: {left_h:.1f} h left")

In [ ]:
# 6. Results summary (the full tables are printed above and saved in the JSON files)
from sphota.build.gate import print_sweep
for f in sorted(glob.glob("bench/results/v3-sweep-medium-*.json")):
    r = json.load(open(f))
    print("=" * 100, "\n", f)
    if "primary_gate" in r:
        print("PRIMARY:", r["primary_gate"])
        for p in r["primary_rows"]:
            print("  ", {k: p[k] for k in ("seed", "coverage", "translation_confident", "cross_model",
                                            "cross_model_ci95", "cross_model_commands", "collapse_max", "passes")})
    else:
        print("INCOMPLETE (session stopped mid-run): rows so far =", len(r["rows"]))

In [ ]:
# 7. Save everything to download (Output tab -> sphota-run3.zip): result JSONs + logs (no embeddings, no models)
!cd /kaggle/working && zip -qj sphota-run3.zip sphota/bench/results/v3-sweep-medium-*.json run3*.log smoke.log 2>/dev/null; ls -la /kaggle/working/sphota-run3.zip
print("Download sphota-run3.zip from the Output tab and upload it to the chat.")

## Reading the result
- **PRIMARY: PASS (confirmed by both replicates)** → the fine code holds across languages and models at ≥ 256 cells on held-out FLORES; the short-command cross-model number says whether that also holds for commands. Next step: an RFC for space `s2`.
- **Otherwise** → the current positioning stands (32 regions across languages, stored canonical vectors for migration, fine codes within one model), and the sweep shows which factor is closest to the bar.
Thresholds: `DECISIONS.md` D-017, D-018; this run: D-050, D-051. Guide: `docs/gpu-run.md`.